In [22]:
!pip install transformers datasets torch scikit-learn rouge-score nltk


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

dataset_path = "/content/drive/MyDrive/samsum-train.csv"  
dataset = pd.read_csv(dataset_path, nrows=1000)

# Verify dataset columns
print(dataset.columns)
documents = dataset['dialogue'].tolist()
summaries = dataset['summary'].tolist()

print(f"Number of Samples: {len(documents)}")
print(f"Example Document: {documents[0][:200]}...")
print(f"Example Summary: {summaries[0]}")


Index(['id', 'dialogue', 'summary'], dtype='object')
Number of Samples: 1000
Example Document: Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)...
Example Summary: Amanda baked cookies and will bring Jerry some tomorrow.


In [25]:
print(type(documents))
print(type(summaries))
print(type(documents[0]))
print(type(summaries[0]))


<class 'list'>
<class 'list'>
<class 'str'>
<class 'str'>


In [26]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')

def label_sentences(documents, summaries, threshold=0.3):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    all_labels = []

    for i, (doc, summary) in enumerate(zip(documents, summaries)):
        if not isinstance(doc, str) or not isinstance(summary, str) or not doc.strip() or not summary.strip():
            print(f"Skipping invalid document or summary at index {i}")
            continue

        try:
            sentences = nltk.sent_tokenize(doc)
            if not sentences:
                print(f"No sentences found in document at index {i}")
                continue

            labels = [
                1 if scorer.score(summary, sent)['rougeL'].fmeasure > threshold else 0
                for sent in sentences
            ]
            all_labels.append((sentences, labels))
        except Exception as e:
            print(f"Error processing document at index {i}: {e}")

    return all_labels



labeled_data = label_sentences(documents, summaries)
print(f"Example Labeled Sentences: {labeled_data[0]}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Example Labeled Sentences: (['Amanda: I baked  cookies.', 'Do you want some?', 'Jerry: Sure!', "Amanda: I'll bring you tomorrow :-)"], [1, 0, 0, 1])


In [28]:
from transformers import BertTokenizer
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_sentences(sentences, labels, max_length=512):
    inputs = tokenizer(sentences, padding=True, truncation=True, max_lengthsent=max_length, return_tensors="pt")
    inputs['labels'] = torch.tensor(labels, dtype=torch.long)
    return inputs


In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
split_ratio = 0.8
train_data = labeled_data[:int(len(labeled_data) * split_ratio)]
val_data = labeled_data[int(len(labeled_data) * split_ratio):]

train_sentences = [data[0] for data in train_data]
train_labels = [data[1] for data in train_data]
val_sentences = [data[0] for data in val_data]
val_labels = [data[1] for data in val_data]

flat_train_sentences = [sent for sents in train_sentences for sent in sents]
flat_train_labels = [label for lbls in train_labels for label in lbls]

flat_val_sentences = [sent for sents in val_sentences for sent in sents]
flat_val_labels = [label for lbls in val_labels for label in lbls]

In [31]:
from torch.utils.data import Dataset
class SummaryDataset(Dataset):
    def __init__(self, sentences, labels, max_length=256):
        self.inputs = tokenize_sentences(sentences, labels, max_length)

    def __len__(self):
        return len(self.inputs['input_ids'])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.inputs.items()}


In [32]:
def tokenize_sentences(sentences, labels, max_length=256):
    inputs = tokenizer(sentences, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    inputs['labels'] = torch.tensor(labels, dtype=torch.long)
    return inputs


In [ ]:
from torch.utils.data import DataLoader 


train_dataset = SummaryDataset(flat_train_sentences, flat_train_labels)
val_dataset = SummaryDataset(flat_val_sentences, flat_val_labels)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [35]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
from torch.utils.data import DataLoader
from transformers import BertForSequenceClassification, BertTokenizer, AdamW
from rouge_score import rouge_scorer
import nltk
import torch
from sklearn.metrics import accuracy_score
import numpy as np


def train_model(model, train_loader, val_loader, optimizer, epochs=3, device='cpu'):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}


            outputs = model(**batch)
            loss = outputs.loss
            total_loss += loss.item()

        
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        
        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader):.4f}")

       
        preds, labels, rouge_scores = evaluate_model(model, val_loader, device)
        avg_rouge = np.mean([score['rougeL'].fmeasure for score in rouge_scores])
        print(f"Validation ROUGE-L: {avg_rouge:.4f}")


def evaluate_model(model, val_loader, device='cpu'):
    model.eval()
    all_preds = []
    all_labels = []
    rouge_scorer_instance = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())

            
            for i, pred in enumerate(preds.cpu().numpy()):
                label = batch['labels'][i].cpu().item()
                rouge_score = rouge_scorer_instance.score(str(label), str(pred))
                rouge_scores.append(rouge_score)

    return all_preds, all_labels, rouge_scores


optimizer = AdamW(model.parameters(), lr=5e-5)


train_model(model, train_loader, val_loader, optimizer, epochs=3, device=device)


Epoch 1, Loss: 0.2564
Validation ROUGE-L: 0.9341
Epoch 2, Loss: 0.2090
Validation ROUGE-L: 0.9341
Epoch 3, Loss: 0.1475
Validation ROUGE-L: 0.9044


In [ ]:
import torch
import nltk


nltk.download('punkt')

def extractive_summarization(document, model, tokenizer, max_sentences=3, max_length=512, device='cpu'):

    sentences = nltk.sent_tokenize(document)
    if not sentences:
        return "Document has no sentences to summarize."

    
    inputs = tokenizer(sentences, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        scores = torch.softmax(logits, dim=1)[:, 1]  

    
    ranked_sentences = sorted(zip(sentences, scores.cpu().numpy()), key=lambda x: x[1], reverse=True)

    
    summary_sentences = [sentence for sentence, _ in ranked_sentences[:max_sentences]]

   
    summary = " ".join(summary_sentences)
    return summary

example_document = "Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of 'intelligent agents': any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals."
summary = extractive_summarization(example_document, model, tokenizer, max_sentences=3, device=device)
print(f"Extractive Summary:\n{summary}")


Extractive Summary:
Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of 'intelligent agents': any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
